# Exporting cleaned Data to SQL Server

In [1]:
from sqlalchemy import create_engine, text
import pandas as pd

In [2]:
df = pd.read_csv('D:/SC & LA/dataco_clean.csv')

In [5]:
df

,payment_type,days_shipping_real,days_shipping_scheduled,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_fname,...,delivery_delay_days,is_on_time,is_discounted,is_profitable,net_revenue,order_year,order_month,order_quarter,order_month_name,processing_days
0,DEBIT,3,4,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,Cally,...,-1,1,1,1,314.640000,2018,1,1,January,3
1,TRANSFER,5,4,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,Irene,...,1,0,1,0,311.360001,2018,1,1,January,5
2,CASH,4,4,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,Gillian,...,0,1,1,0,309.719999,2018,1,1,January,4
3,DEBIT,3,4,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,Tana,...,-1,1,1,1,304.809999,2018,1,1,January,3
4,PAYMENT,2,4,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,Orli,...,-2,1,1,1,298.250000,2018,1,1,January,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180514,CASH,4,4,Shipping on time,0,45,Fishing,Brooklyn,EE. UU.,Maria,...,0,1,0,1,399.980011,2016,1,1,January,4
180515,DEBIT,3,2,Late delivery,1,45,Fishing,Bakersfield,EE. UU.,Ronald,...,1,0,1,0,395.980011,2016,1,1,January,3
180516,TRANSFER,5,4,Late delivery,1,45,Fishing,Bristol,EE. UU.,John,...,1,0,1,1,391.980011,2016,1,1,January,5
180517,PAYMENT,3,4,Advance shipping,0,45,Fishing,Caguas,Puerto Rico,Mary,...,-1,1,1,1,387.980011,2016,1,1,January,3


In [7]:
from sqlalchemy import create_engine

engine = create_engine(
    "mssql+pyodbc://@MANIKANTAVEJAND\\SQLEXPRESS/DataCoDB"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

In [9]:
#dim_product
dim_product = df[[
    'product_id','product_name','product_price',
    'product_category_id','category_id','category_name',
    'department_id','department_name'
]].drop_duplicates(subset='product_id').reset_index(drop=True)

In [11]:
# dim_customer
dim_customer = df[[
    'customer_id','customer_fname','customer_lname',
    'customer_segment','customer_city',
    'customer_state','customer_country'
]].drop_duplicates(subset='customer_id').reset_index(drop=True)

In [13]:
# dim_date — one row per calendar date
all_dates = pd.date_range('2015-01-01', '2018-01-31')
dim_date = pd.DataFrame({
    'date_key':    all_dates.strftime('%Y%m%d').astype(int),
    'full_date':   all_dates.date,
    'year':        all_dates.year,
    'quarter':     all_dates.quarter,
    'month':       all_dates.month,
    'month_name':  all_dates.strftime('%B'),
    'week_num':    all_dates.isocalendar().week,
    'day_of_week': all_dates.day_name(),
    'is_weekend' : all_dates.dayofweek.isin([5,6]).astype(int)
})

In [15]:
# fact_orders — the flat transactional table
fact_orders = df[[
    'order_item_id','order_id','customer_id','product_id',
    'order_date','shipping_date',
    'payment_type','order_status','delivery_status',
    'shipping_mode','market','order_region',
    'order_country','order_city','order_state',
    'latitude','longitude',
    'days_shipping_real','days_shipping_scheduled',
    'late_delivery_risk','delivery_delay_days','is_on_time',
    'item_quantity','sales','item_discount',
    'item_discount_rate','order_item_total',
    'order_profit','item_profit_ratio',
    'net_revenue','is_profitable','is_discounted',
    'processing_days'
]]

In [17]:
# Save dimension tables
dim_customer.to_csv("dim_customer.csv", index=False)

dim_product.to_csv("dim_product.csv", index=False)

dim_date.to_csv("dim_date.csv", index=False)

# Save fact table
fact_orders.to_csv("fact_orders.csv", index=False)

In [19]:
dim_customer.to_csv("D:/SC & LA/dim_customer.csv", index=False)

dim_product.to_csv("D:/SC & LA/dim_product.csv", index=False)

dim_date.to_csv("D:/SC & LA/dim_date.csv", index=False)

fact_orders.to_csv("D:/SC & LA/fact_orders.csv", index=False)